# Experiment and evaluate Olist freight models on Azure ML

This adapts the cloned Microsoft Learn **classification AutoML + MLflow** lab to **Olist freight regression**. Azure AutoML tries several regressors; separate Azure ML command jobs train Ridge and gradient boosting and log validation metrics with MLflow. The test period is not used to choose a model.

Open from this exercise folder in the cloned repository and select **Python (Olist Azure ML)**. Run `python scripts/bootstrap.py --subscription ... --resource-group ... --workspace ...` from the repo root first (see root README); it extracts the shared CSVs, installs the kernel, and writes ignored `config.json`. Sign in with `az login`; Azure CLI with the `ml` extension must work. Data uploads and serverless cloud jobs can incur charges. No jobs are submitted by default.

Reference: [Microsoft Learn lab](https://microsoftlearning.github.io/mslearn-mlops/docs/01-experiment-evaluate-models.html). Do not run its `infra/setup.sh`: that script provisions a separate workspace and computes and uploads diabetes data.

## 1. Locate the lab and choose the existing Azure ML workspace

Read the nonsecret workspace identifiers from the repository's ignored `config.json`. Every Azure CLI call explicitly targets that workspace; nothing changes your global Azure CLI defaults. The ZIP marker finds the cloned repository even if Jupyter starts in a parent directory.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

marker = Path('datasets/olist/Ecommerce Dataset.zip')
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / marker).is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the cloned repo containing datasets/olist/Ecommerce Dataset.zip')
LAB = ROOT / 'exercises' / '05-freight-experimentation'
RAW = ROOT / 'datasets' / 'olist' / 'Ecommerce Dataset'
if not (LAB / 'prepare.py').is_file() or not (RAW / 'olist_order_items_dataset.csv').is_file():
    raise FileNotFoundError('Run scripts/bootstrap.py from the repo root to extract the shared Olist CSVs')
config = json.loads((ROOT / 'config.json').read_text(encoding='utf-8'))
SUBSCRIPTION = config['subscription_id']
RESOURCE_GROUP = config['resource_group']
WORKSPACE = config['workspace_name']
AZ_CLI = shutil.which('az')
assert AZ_CLI is not None, 'Install Azure CLI and its ml extension on the notebook host.'
run_ids_file = LAB / '.freight-run-ids.json'
run_ids = json.loads(run_ids_file.read_text(encoding='utf-8')) if run_ids_file.is_file() else {}
UPLOAD_DATA = False  # Opt in only after reviewing prepared data and workspace costs.
SUBMIT_JOBS = False  # Opt in to the three billable training jobs; recorded IDs prevent duplicates.
SUBMIT_EVALUATION = False  # Separate opt-in after selecting a validation winner.
print('Experiment files:', LAB.resolve())
print('Raw Olist data:', RAW.resolve())

def az_ml(*args):
    cmd = [AZ_CLI, 'ml', *map(str, args), '-g', RESOURCE_GROUP, '-w', WORKSPACE,
           '--subscription', SUBSCRIPTION, '--only-show-errors', '-o', 'json']
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f'Azure CLI failed: {result.stderr or result.stdout}')
    return json.loads(result.stdout) if result.stdout.strip() else {}

## 2. Check that the workspace is reachable

This is a read-only request. If it fails, check `az login` and your workspace access before uploading any data.

In [ ]:
workspace = subprocess.run(
    [AZ_CLI, 'ml', 'workspace', 'show', '-n', WORKSPACE, '-g', RESOURCE_GROUP,
     '--subscription', SUBSCRIPTION, '--query', '{name:name,location:location}', '-o', 'json'],
    text=True, capture_output=True, check=True,
)
print('Workspace:', workspace.stdout.strip())

## 3. Build one row per order item and split by purchase date

The preparation script joins Olist items, orders, products, sellers, customers, category translations and ZIP-prefix coordinates. It calculates distance, volume, state match, cart counts and purchase month. It writes **train before April 2018**, **validation April-May 2018** and **test June-August 2018**. Delivery outcomes and order IDs are not exported as model features.

In [ ]:
subprocess.run([sys.executable, str(LAB / 'prepare.py'), '--raw-data', str(RAW),
                '--output-dir', str(LAB / 'data')], check=True)

## 4. Check the processed data before it leaves your machine

Each CSV contains the target `freight_value` plus only checkout-time predictors. The test CSV is reserved for later evaluation and must not be used to choose AutoML settings or the manual model. The `MLTable` file beside each CSV explains how Azure ML will read it.

In [ ]:
import pandas as pd
for split in ['train', 'validation', 'test']:
    frame = pd.read_csv(LAB / 'data' / split / 'orders.csv')
    print(split, ':', len(frame), 'rows;', len(frame.columns), 'columns;',
          'missing targets:', frame.freight_value.isna().sum())
    assert 'order_id' not in frame and 'order_delivered_customer_date' not in frame
assert (LAB / 'data' / 'train' / 'MLTable').is_file()

## 5. Upload the three versioned MLTable data assets

**Opt-in cloud action:** set `UPLOAD_DATA = True` in step 1 to upload missing splits to your workspace as version 1. Existing assets are reused, never overwritten. If a lookup fails for any reason other than a missing asset, stop rather than trying to create it. Publish changed data under a new version and update the job inputs.

In [ ]:
for split in ['train', 'validation', 'test']:
    name = f'olist-freight-{split}'
    cmd = [AZ_CLI, 'ml', 'data', 'show', '-n', name, '-v', '1', '-g', RESOURCE_GROUP,
           '-w', WORKSPACE, '--subscription', SUBSCRIPTION, '--only-show-errors', '-o', 'json']
    existing = subprocess.run(cmd, capture_output=True, text=True)
    if existing.returncode == 0:
        print('Already registered:', name, 'version 1')
    elif not any(word in (existing.stderr + existing.stdout).lower()
                 for word in ('notfound', 'not found', 'does not exist')):
        raise RuntimeError(f'Cannot check data asset {name}: {existing.stderr or existing.stdout}')
    elif not UPLOAD_DATA:
        print('Missing:', name, 'version 1; set UPLOAD_DATA = True to upload')
    else:
        asset = az_ml('data', 'create', '-f', LAB / 'data' / f'{split}.yml')
        print('Created:', asset['name'], 'version', asset['version'])

## 6. Submit AutoML regression in Azure ML

**Billable cloud action, disabled by default:** set `SUBMIT_JOBS = True` in step 1 only when ready. AutoML tries at most five regressors with a 90-minute total limit and one concurrent trial. It chooses on the earlier validation period and optimizes normalized RMSE; examine MAE in reais in Studio as well. Job IDs are saved in the ignored `.freight-run-ids.json`; you can also populate that file with `automl`, `ridge`, and `boosting` job names from your own workspace to reconnect. Never use another workspace's run IDs.

In [ ]:
def job_or_submit(key, *args):
    if run_ids.get(key):
        job = az_ml('job', 'show', '-n', run_ids[key])
    elif SUBMIT_JOBS:
        job = az_ml('job', 'create', *args)
        run_ids[key] = job['name']
        run_ids_file.write_text(json.dumps(run_ids, indent=2) + '\n', encoding='utf-8')
    else:
        print(key, 'not submitted; set SUBMIT_JOBS = True to opt in')
        return None
    print(key, 'job:', job['name'], job.get('studio_url', ''))
    return job

automl_job = job_or_submit('automl', '-f', LAB / 'jobs' / 'automl-regression.yml')

## 7. Submit a separate MLflow-tracked Ridge regression job

**Billable cloud action:** this uses a curated scikit-learn environment on serverless CPU compute. The training script fits a median baseline and Ridge, logs MAE/RMSE and a prediction plot with MLflow, and saves an MLflow-format model in job outputs. This is the counterpart of the official *Track model training with MLflow* notebook, but the training itself runs in Azure ML.

In [ ]:
ridge_job = job_or_submit('ridge', '-f', LAB / 'jobs' / 'track-mlflow.yml')

## 8. Change one parameter and submit a second MLflow run

**Billable cloud action:** reuse the same YAML and datasets, changing only the algorithm to gradient boosting. Separate Azure ML jobs make it easy to compare their validation metrics in Studio.

In [ ]:
boost_job = job_or_submit('boosting', '-f', LAB / 'jobs' / 'track-mlflow.yml',
                          '--set', 'inputs.algorithm=boosting',
                          'display_name=olist-freight-boosting')

## 9. Check job status and compare validation runs

This cell only **reads** job status; it does not keep polling or start another job. Wait for all jobs to finish, then rerun it. In Azure ML Studio, open Jobs -> `olist-freight-automl` for child-model comparisons and data guardrails; open Jobs -> `olist-freight-mlflow` for `validation_mae_brl`, the median baseline and the prediction plots. The smallest validation MAE is best for an R$ error objective; do not inspect test outcomes while choosing.

In [ ]:
for name in [job['name'] for job in (automl_job, ridge_job, boost_job) if job]:
    job = az_ml('job', 'show', '-n', name)
    print(name, job['status'], job.get('studio_url', ''))

## 10. Evaluate the selected manual model once on future orders

**Billable cloud action, separately disabled by default:** compare validation metrics first, then set `selected_algorithm` to `ridge` or `boosting` and `SUBMIT_EVALUATION = True` in step 1. An existing evaluation job ID in `.freight-run-ids.json` is reused without registering another model. The selected model version is passed explicitly to the test job; AutoML's winner can be investigated separately in Studio.

In [ ]:
selected_algorithm = None  # Choose 'ridge' or 'boosting' after comparing validation MAE.
if run_ids.get('evaluation'):
    evaluation_job = az_ml('job', 'show', '-n', run_ids['evaluation'])
    print('Existing evaluation job:', evaluation_job['name'])
elif not SUBMIT_EVALUATION:
    print('Evaluation disabled; set SUBMIT_EVALUATION = True in step 1 after selecting a winner')
elif selected_algorithm not in ('ridge', 'boosting') or not run_ids.get(selected_algorithm):
    raise ValueError('Select a completed ridge or boosting job from your workspace first')
else:
    selected_job = run_ids[selected_algorithm]
    if az_ml('job', 'show', '-n', selected_job)['status'] != 'Completed':
        print('Wait for the selected training job to complete; no test job was submitted.')
    else:
        model_path = (f'azureml://jobs/{selected_job}/outputs/artifacts/paths/'
                      f'outputs/{selected_algorithm}-model/')
        if run_ids.get('evaluation_model_version'):
            if run_ids.get('evaluation_source') != selected_job:
                raise ValueError('Stored evaluation model is from a different training job')
            version = run_ids['evaluation_model_version']
        else:
            registered = az_ml('model', 'create', '--name', 'olist-freight-regression-exp',
                               '--type', 'mlflow_model', '--path', model_path)
            version = registered['version']
            run_ids.update(evaluation_model_version=version, evaluation_source=selected_job)
            run_ids_file.write_text(json.dumps(run_ids, indent=2) + '\n', encoding='utf-8')
        evaluation_job = az_ml('job', 'create', '-f', LAB / 'jobs' / 'evaluate.yml',
                               '--set', f'inputs.trained_model.path=azureml:olist-freight-regression-exp:{version}')
        run_ids['evaluation'] = evaluation_job['name']
        run_ids_file.write_text(json.dumps(run_ids, indent=2) + '\n', encoding='utf-8')
        print('Held-out evaluation job:', evaluation_job['name'])

## What is still needed for a production decision?

The June-August **test asset stays held out until Step 10**. Examine error by product size, state and purchase month; retrain on more recent data before considering a live quote. Azure ML compute and data storage may incur costs: the serverless jobs end when complete, but review the workspace's storage and any separately running compute instances when finished.